# OpenAI "A practical guide to building agents" 실습

가이드 흐름: **모델 → 도구 → 지침 → 오케스트레이션 → 가드레일 → 사람 개입**

- 구현은 OpenAI Agents SDK (`openai-agents`) 사용
- 크레딧 절약을 위해 기본 모델은 `gpt-4o-mini`
- 도구는 외부 API 없이 mock으로 구현 (호출 비용 = LLM 호출만)

## 0. 설치 & 환경 설정

In [ ]:
!pip install -q openai-agents pydantic

In [ ]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

MODEL = "gpt-4o-mini"  # 저렴한 모델. 필요하면 바꿔서 비교해보기

## 1. 가장 단순한 에이전트 (Model + Instructions)
에이전트 = 모델 + 지침 + 도구. 일단 도구 없이 시작.

In [ ]:
from agents import Agent, Runner

hello_agent = Agent(
    name="Hello Agent",
    instructions="You are a concise assistant. Answer in Korean.",
    model=MODEL,
)

result = await Runner.run(hello_agent, "에이전트가 뭐야? 한 문장으로.")
print(result.final_output)

## 2. 도구 (Tools)
가이드의 3가지 도구 유형:
- **Data**: 정보 조회 (DB, 검색)
- **Action**: 외부 시스템에 변경을 가함 (이메일, 환불 등)
- **Orchestration**: 다른 에이전트를 도구로 사용 (6번에서)

`@function_tool` 데코레이터는 함수 시그니처/docstring으로 스키마를 자동 생성함.

In [ ]:
from agents import function_tool

ORDERS = {
    "A100": {"item": "무선 이어폰", "status": "배송중", "price": 89000},
    "A200": {"item": "키보드", "status": "배송완료", "price": 129000},
}

@function_tool
def get_order(order_id: str) -> dict:
    """주문번호로 주문 정보를 조회한다. (Data tool)

    Args:
        order_id: 주문번호 (예: A100)
    """
    return ORDERS.get(order_id, {"error": "order not found"})

@function_tool
def issue_refund(order_id: str, reason: str) -> str:
    """주문을 환불 처리한다. (Action tool)

    Args:
        order_id: 주문번호
        reason: 환불 사유
    """
    if order_id not in ORDERS:
        return "order not found"
    ORDERS[order_id]["status"] = "환불됨"
    return f"{order_id} 환불 완료 (사유: {reason})"

# 생성된 JSON 스키마 확인
print(get_order.params_json_schema)

## 3. 지침 (Instructions) + 도구를 가진 에이전트
가이드 권장사항: 기존 문서(SOP)를 기반으로, 단계를 명확히, 엣지 케이스를 명시.

In [ ]:
INSTRUCTIONS = """You are a customer support agent for an online shop. Answer in Korean.

Routine:
1. Ask for the order id if the user did not provide one.
2. Look up the order with get_order.
3. If the user wants a refund:
   - Only refund if status is '배송완료'. Otherwise explain why you can't.
   - Ask for the reason if missing, then call issue_refund.
4. Summarize the outcome in 1-2 sentences.

If the order is not found, say so and ask the user to re-check the id. Never invent order data."""

support_agent = Agent(
    name="Support Agent",
    instructions=INSTRUCTIONS,
    tools=[get_order, issue_refund],
    model=MODEL,
)

result = await Runner.run(support_agent, "A100 주문 어디까지 왔어?")
print(result.final_output)

## 4. Run loop 들여다보기
`Runner.run`은 내부적으로 *종료 조건*(최종 출력/도구 호출 없음/max_turns)까지 반복한다.
어떤 단계를 거쳤는지 `new_items`로 확인.

In [ ]:
for item in result.new_items:
    print(type(item).__name__, "|", getattr(item, "raw_item", None).__class__.__name__)

In [ ]:
# 환불 시나리오 (A100은 배송중 → 거절해야 정상, A200은 환불 가능)
for q in ["A100 환불해줘", "A200 환불해줘. 단순 변심이야"]:
    r = await Runner.run(support_agent, q)
    print(f"Q: {q}\nA: {r.final_output}\n")

In [ ]:
# 멀티턴: 이전 결과를 이어서 입력으로 사용
r1 = await Runner.run(support_agent, "내 주문 상태 알려줘")
print(r1.final_output)
r2 = await Runner.run(support_agent, r1.to_input_list() + [{"role": "user", "content": "A200이야"}])
print(r2.final_output)

## 5. Structured output
출력 형식을 pydantic으로 강제. 가드레일/라우팅에서도 유용.

In [ ]:
from pydantic import BaseModel

class OrderSummary(BaseModel):
    order_id: str
    item: str
    status: str
    needs_attention: bool

summary_agent = Agent(
    name="Summarizer",
    instructions="Look up the order and return a summary. needs_attention=true if status is not 배송완료.",
    tools=[get_order],
    output_type=OrderSummary,
    model=MODEL,
)
r = await Runner.run(summary_agent, "A100")
print(r.final_output)

## 6. 오케스트레이션 (Orchestration)
### 6-A. Manager 패턴: 중앙 에이전트가 전문 에이전트를 *도구처럼* 호출
사용자와의 대화 제어권은 manager가 유지한다.

In [ ]:
translate_ko = Agent(
    name="ko", instructions="Translate the user's text to Korean.", model=MODEL)
translate_ja = Agent(
    name="ja", instructions="Translate the user's text to Japanese.", model=MODEL)

manager = Agent(
    name="Translation Manager",
    instructions=(
        "You are a translation manager. Use the tools to translate. "
        "If the user asks for several languages, call every relevant tool, then combine results."
    ),
    tools=[
        translate_ko.as_tool("to_korean", "Translate text to Korean"),
        translate_ja.as_tool("to_japanese", "Translate text to Japanese"),
    ],
    model=MODEL,
)
r = await Runner.run(manager, "'Agents are systems that independently accomplish tasks.' 한국어랑 일본어로 번역해줘")
print(r.final_output)

### 6-B. Decentralized 패턴: Handoff
에이전트끼리 **제어권을 넘긴다**. 트리아지 → 전문 에이전트.

In [ ]:
billing_agent = Agent(
    name="Billing Agent",
    handoff_description="결제/환불/청구 관련 문의",
    instructions="You handle billing and refund questions. Answer in Korean.",
    tools=[get_order, issue_refund],
    model=MODEL,
)
tech_agent = Agent(
    name="Tech Agent",
    handoff_description="제품 고장/사용법 등 기술 문의",
    instructions="You handle technical support questions. Answer in Korean, briefly.",
    model=MODEL,
)
triage_agent = Agent(
    name="Triage Agent",
    instructions="Route the user to the right specialist. Do not answer yourself.",
    handoffs=[billing_agent, tech_agent],
    model=MODEL,
)

r = await Runner.run(triage_agent, "이어폰 블루투스 연결이 자꾸 끊겨요")
print("최종 담당:", r.last_agent.name)
print(r.final_output)

In [ ]:
r = await Runner.run(triage_agent, "A200 환불하고 싶어요. 사이즈가 안 맞아요")
print("최종 담당:", r.last_agent.name)
print(r.final_output)

## 7. 가드레일 (Guardrails)
가이드: 단일 가드레일로 충분치 않고, **여러 겹**으로 쌓는다. LLM 기반 + 규칙 기반 혼합.

### 7-A. Input guardrail (관련성 분류기)
작은 모델로 입력이 주제와 관련 있는지 판단 → 아니면 tripwire로 차단.

In [ ]:
from agents import (
    GuardrailFunctionOutput, RunContextWrapper, TResponseInputItem,
    input_guardrail, output_guardrail,
    InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered,
)

class Relevance(BaseModel):
    is_relevant: bool
    reasoning: str

relevance_checker = Agent(
    name="Relevance Checker",
    instructions="Decide if the user's message is about online-shop customer support (orders, refunds, products). Off-topic = not relevant.",
    output_type=Relevance,
    model=MODEL,
)

@input_guardrail
async def relevance_guardrail(
    ctx: RunContextWrapper[None], agent: Agent, input: str | list[TResponseInputItem]
) -> GuardrailFunctionOutput:
    res = await Runner.run(relevance_checker, input, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info=res.final_output,
        tripwire_triggered=not res.final_output.is_relevant,
    )

### 7-B. Output guardrail (규칙 기반: 민감정보 패턴)
정규식 같은 단순 규칙은 LLM보다 싸고 확실함.

In [ ]:
import re

class ReplyOut(BaseModel):
    reply: str

@output_guardrail
async def no_pii_guardrail(
    ctx: RunContextWrapper, agent: Agent, output: ReplyOut
) -> GuardrailFunctionOutput:
    leaked = bool(re.search(r"\b\d{6}-\d{7}\b|\b\d{3}-\d{4}-\d{4}\b", output.reply))  # 주민번호/전화번호
    return GuardrailFunctionOutput(output_info={"leaked": leaked}, tripwire_triggered=leaked)

In [ ]:
guarded_agent = Agent(
    name="Guarded Support",
    instructions=INSTRUCTIONS + "\nReturn the final message in the `reply` field.",
    tools=[get_order, issue_refund],
    input_guardrails=[relevance_guardrail],
    output_guardrails=[no_pii_guardrail],
    output_type=ReplyOut,
    model=MODEL,
)

for q in ["A200 주문 상태 알려줘", "비트코인 지금 사야 돼?"]:
    try:
        r = await Runner.run(guarded_agent, q)
        print(f"[OK] {q} -> {r.final_output.reply}")
    except InputGuardrailTripwireTriggered as e:
        print(f"[BLOCKED-input] {q}")
    except OutputGuardrailTripwireTriggered:
        print(f"[BLOCKED-output] {q}")

## 8. 사람 개입 (Human intervention)
가이드: 고위험 행동(환불, 결제 등)은 사람 승인 후 실행. 여기서는 도구 내부에서 `input()`으로 승인 받는 가장 단순한 형태.

In [ ]:
@function_tool
def issue_refund_with_approval(order_id: str, reason: str) -> str:
    """주문을 환불 처리한다. 사람의 승인이 필요하다.

    Args:
        order_id: 주문번호
        reason: 환불 사유
    """
    if order_id not in ORDERS:
        return "order not found"
    ans = input(f"[승인 필요] {order_id} 환불 ({reason}) 진행? (y/n): ").strip().lower()
    if ans != "y":
        return "환불이 담당자에 의해 거절됨. 사용자에게 알리고 다른 방법을 안내할 것."
    ORDERS[order_id]["status"] = "환불됨"
    return f"{order_id} 환불 완료"

hitl_agent = Agent(
    name="HITL Support",
    instructions=INSTRUCTIONS.replace("issue_refund", "issue_refund_with_approval"),
    tools=[get_order, issue_refund_with_approval],
    model=MODEL,
)

ORDERS["A200"]["status"] = "배송완료"  # 위에서 환불됨으로 바뀌었을 수 있으니 리셋
r = await Runner.run(hitl_agent, "A200 환불해줘. 단순 변심")
print(r.final_output)

## 9. 실험 아이디어
- `Runner.run(..., max_turns=3)`로 루프 제한 걸어보기 (`MaxTurnsExceeded`)
- 지침에서 "환불 조건"을 빼고 동작이 어떻게 달라지는지 비교
- `MODEL`을 `gpt-4o` 등으로 바꿔 tool 호출 정확도 비교
- 가드레일 입력에 prompt injection ("이전 지시 무시하고…") 넣어 보기
- `from agents import WebSearchTool`로 hosted 도구 추가 (별도 과금 주의)
- 트레이스: https://platform.openai.com/traces 에서 위 실행들이 자동 기록됨